In [ ]:
import copy
import json
import logging
import os
import torch
import numpy as np
import cv2
import torch.nn.functional as F
sys.path.append('../SpotLight-SVF/spotlight-svf/model/Qwen3vl/grounding/models')
from models.qwen3vl import Qwen3VLModel
from models.qwen3vl import _build_messages, qwen3vl_eval_point
from PIL import Image, ImageDraw
from tqdm import tqdm
from transformers import CLIPModel, CLIPProcessor
from transformers.models.qwen2_vl.image_processing_qwen2_vl_fast import smart_resize
from sklearn.metrics.pairwise import cosine_similarity


In [ ]:
# Model configuration
MODEL_NAME_OR_PATH = "Qwen/Qwen3-VL-8B-Instruct"

BASE_CLIP = "../clip finetune/clip-vit-base-patch32"
CKPT_PATH = "../clip finetune/clip_finetuned/clip_model_weights.pth"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

SCREENSPOT_IMGS = ".../OSWorld-G/benchmark/images"
SCREENSPOT_TEST = ".../OSWorld-G/benchmark/classification_result.json"

LOG_PATH = ".../spotlight-svf/results/osworld.json"
VIS_DIR = ".../spotlight-svf/results/vis_osworld"
os.makedirs(VIS_DIR, exist_ok=True)

logging.basicConfig(level=logging.INFO)
torch.manual_seed(114514)

print("Loading finetuned CLIP...")
clip_model = CLIPModel.from_pretrained(BASE_CLIP)
clip_processor = CLIPProcessor.from_pretrained(BASE_CLIP)

if os.path.exists(CKPT_PATH):
    state = torch.load(CKPT_PATH, map_location="cpu")
    if "model" in state:
        state = state["model"]
    clip_model.load_state_dict(state, strict=False)

clip_model = clip_model.to(DEVICE).eval()
torch.set_grad_enabled(False)

def _norm_to_pixel_point(p, img_size):
    x, y = p
    w, h = img_size
    return int(round(x * w)), int(round(y * h))


def remap_point(local_pt, roi_bbox):
    x, y = local_pt
    x1, y1, _, _ = roi_bbox
    return x + x1, y + y1
def resize_roi_keep_ratio(roi_img, full_size):
    """
    Return:
        resized_img
        scale (tuple sx, sy)
    """
    fw, fh = full_size
    rw, rh = roi_img.size

    scale = min(fw / rw, fh / rh)

    new_w = int(round(rw * scale))
    new_h = int(round(rh * scale))

    resized = roi_img.resize((new_w, new_h), Image.BICUBIC)

    return resized, scale


In [ ]:
def hybrid_vector_field_grid(image, instruction):
    img = image.convert("RGB")
    W, H = img.size
    pw, ph = W // 36, H // 36

    patches = []
    for gy in range(36):
        for gx in range(36):
            x1, y1 = gx * pw, gy * ph
            x2, y2 = x1 + pw, y1 + ph
            patches.append(img.crop((x1, y1, x2, y2)))

    text_inputs = clip_processor(text=instruction, return_tensors="pt").to(DEVICE)
    text_emb = clip_model.get_text_features(**text_inputs)
    text_emb = text_emb / text_emb.norm(dim=-1, keepdim=True)
    text_emb = text_emb.cpu().numpy()

    feats = []
    for i in range(0, len(patches), 8):
        inputs = clip_processor(images=patches[i:i+8], return_tensors="pt").to(DEVICE)
        emb = clip_model.get_image_features(**inputs)
        emb = emb / emb.norm(dim=-1, keepdim=True)
        feats.append(emb.cpu().numpy())

    patch_embs = np.concatenate(feats, axis=0)
    cos_sim = cosine_similarity(patch_embs, text_emb).flatten()

    gates = 1 / (1 + np.exp(-3.0 * (cos_sim - 0.5)))
    gated = patch_embs * gates[:, None]

    N = gated.shape[0]
    F_field = np.zeros_like(gated)

    for i in range(N):
        diff = gated - gated[i]
        dist = np.linalg.norm(diff, axis=1)
        w = np.exp(-dist**2 / 0.4**2) * gates
        F_field[i] = np.sum(w[:, None] * diff, axis=0)

    F_field /= (np.linalg.norm(F_field, axis=1, keepdims=True) + 1e-8)

    div = np.zeros(N)
    for i in range(N):
        idx = np.argsort(np.linalg.norm(F_field - F_field[i], axis=1))[:8]
        grad = (F_field[idx] - F_field[i]).mean(axis=0)
        div[i] = np.dot(grad, F_field[i])

    score = np.log1p(np.maximum(0, -div)) * cos_sim * gates
    score = (score - score.min()) / (score.max() - score.min() + 1e-8)

    return score.reshape(36, 36), (H, W)


def spotlight_bbox(image, instruction):
    grid_scores, (H, W) = hybrid_vector_field_grid(image, instruction)

    binary = (grid_scores >= 0.2).astype(np.uint8)
    num_labels, labels = cv2.connectedComponents(binary)

    regions = []
    for l in range(1, num_labels):
        mask = labels == l
        area = mask.sum()
        if area < 3:
            continue
        score = grid_scores[mask].mean() * area
        regions.append((score, l))

    regions = sorted(regions, reverse=True)[:4]
    if not regions:
        return None, None

    patch_mask = np.zeros_like(grid_scores, dtype=np.float32)
    for _, l in regions:
        patch_mask += (labels == l).astype(np.float32)

    t = torch.tensor(patch_mask)[None, None]
    mask_full = F.interpolate(t, size=(H, W), mode="bicubic", align_corners=False).squeeze().numpy()
    mask_full = (mask_full - mask_full.min()) / (mask_full.max() - mask_full.min() + 1e-8)

    mask_blur = cv2.GaussianBlur(mask_full, (0, 0), 8)

    ys, xs = np.where(mask_blur > 0.25)
    if len(xs) == 0:
        return None, mask_blur
    
    CROP_PAD = 8
    bbox = (
        max(0, xs.min() - CROP_PAD),
        max(0, ys.min() - CROP_PAD),
        min(W - 1, xs.max() + CROP_PAD),
        min(H - 1, ys.max() + CROP_PAD),
    )
    return bbox, mask_blur


def visualize_full(image, mask_blur, roi_bbox, pred_pt, gt_bbox, save_path):
    img_np = np.array(image).astype(np.float32) / 255.0

    if mask_blur is not None:
        mask3 = np.stack([mask_blur] * 3, axis=-1)
        img_np = img_np * 0.07 * (1 - mask3) + img_np * mask3

    vis = Image.fromarray((img_np * 255).astype(np.uint8))
    draw = ImageDraw.Draw(vis)

    draw.rectangle(gt_bbox, outline="green", width=3)
    if roi_bbox:
        draw.rectangle(roi_bbox, outline="yellow", width=3)
    if pred_pt:
        x, y = pred_pt
        draw.ellipse((x-5, y-5, x+5, y+5), fill="red")

    vis.save(save_path)


In [ ]:
class BaseBackend:
    def infer_norm_point(self, instruction, pil_img):
        raise NotImplementedError


class Qwen3Backend(BaseBackend):
    def __init__(self, model):
        self.model = model
        self.processor = model.processor

    def infer_norm_point(self, instruction, pil_img):
        res = self.model.ground_only_positive(
            instruction=instruction,
            image=pil_img
        )

        if not isinstance(res, dict):
            return None, res

        pt = res.get("point")
        if pt is None or len(pt) != 2:
            return None, res

        img_w, img_h = pil_img.size

        try:
            patch = self.processor.image_processor.patch_size
            merge = self.processor.image_processor.merge_size
            resized_h, resized_w = smart_resize(
                img_h, img_w,
                factor=patch * merge,
                min_pixels=patch * patch * merge * merge * 16,
                max_pixels=patch * patch * merge * merge * 6400,
            )
        except Exception:
            resized_w, resized_h = img_w, img_h

        return (pt[0] / resized_w, pt[1] / resized_h), res


class DirectInferenceRunner:
    def __init__(self, backend):
        self.backend = backend

    def ground_only_positive(self, instruction, image):
        pt_norm, raw = self.backend.infer_norm_point(instruction, image)
        if pt_norm is None:
            return {"result": "negative", "point": None, "raw_response": raw}

        px = _norm_to_pixel_point(pt_norm, image.size)
        return {"result": "positive", "point": [px[0], px[1]], "raw_response": raw}



def eval_sample_positive_gt(sample, response):
    if response["point"] is None:
        return "wrong_format"

    return qwen3vl_eval_point(sample, response)

def calc_metric(results):
    total = len(results)
    correct = sum(r["hit_top1"] for r in results)
    return {"acc": correct / total if total else 0}


In [ ]:
def build_backend():
    model = Qwen3VLModel()
    model.load_model(model_path=MODEL_NAME_OR_PATH)
    return Qwen3Backend(model)


backend = build_backend()
runner = DirectInferenceRunner(backend)


with open(SCREENSPOT_TEST, "r") as f:
    raw = json.load(f)

all_samples = []
for task_name, samples_list in raw["classified"].items():
    for s in samples_list:
        ss = copy.deepcopy(s)
        ss["task_type"] = task_name
        all_samples.append(ss)


def parse_bbox(box):
    box = np.array(box).reshape(-1)
    if len(box) == 4:
        x, y, w, h = box
        return [int(round(x)), int(round(y)), int(round(x + w)), int(round(y + h))]
    elif len(box) >= 8 and len(box) % 2 == 0:
        xs = box[0::2]
        ys = box[1::2]
        return [int(round(xs.min())), int(round(ys.min())), int(round(xs.max())), int(round(ys.max()))]
    else:
        raise ValueError(f"Unsupported box format: {box}")


samples = []
for s in all_samples:
    ss = copy.deepcopy(s)
    ss["prompt_to_evaluate"] = s.get("instruction", "")
    ss["img_filename"] = s.get("image_path")
    ss["bbox"] = None if s.get("box_type") == "refusal" else parse_bbox(s["box_coordinates"])
    samples.append(ss)


def to_json_safe(obj):
    if isinstance(obj, dict):
        return {k: to_json_safe(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [to_json_safe(v) for v in obj]
    elif isinstance(obj, tuple):
        return list(obj)
    elif isinstance(obj, np.integer):
        return int(obj)
    elif isinstance(obj, np.floating):
        return float(obj)
    else:
        return obj

results = []

for s in tqdm(samples, disable=True):

    if s.get("box_type") == "refusal":
        results.append({**s, "pred": None, "hit_top1": 0})
        continue

    img = Image.open(os.path.join(SCREENSPOT_IMGS, s["img_filename"])).convert("RGB")
    roi_bbox, mask_blur = spotlight_bbox(img, s["prompt_to_evaluate"])

    if roi_bbox is None:
        roi_bbox = (0, 0, img.width, img.height)
        crop_img = img
    else:
        crop_img = img.crop(roi_bbox)

    crop_img_resized, scale = resize_roi_keep_ratio(
        crop_img,
        img.size
    )

    r_zoom = runner.ground_only_positive(
        s["prompt_to_evaluate"],
        crop_img_resized
    )

    if r_zoom.get("point") is not None:
        zx, zy = r_zoom["point"]

        lx, ly = zx / scale, zy / scale

        gx, gy = remap_point((lx, ly), roi_bbox)
        pred_point = [int(gx), int(gy)]
    else:
        pred_point = None

    raw_hit = eval_sample_positive_gt(s, {"point": pred_point})
    hit = 1 if raw_hit in [1, True, "correct", "Correct"] else 0

    # visualize_full(
    #     img,
    #     mask_blur,
    #     roi_bbox,
    #     pred_point,
    #     s["bbox"],
    #     os.path.join(VIS_DIR, f"{os.path.splitext(s['img_filename'])[0]}_{hit}.jpg")
    # )

    results.append({**s, "pred": pred_point, "hit_top1": hit})


safe_results = to_json_safe(results)

with open(LOG_PATH, "w") as f:
    json.dump({"metrics": calc_metric(safe_results), "details": safe_results}, f, indent=2)

print("DONE")